# 01: Data Loading, Audit & Preprocessing Pipeline

**RetailPulse AI Platform** • *Zidio Development Industry Project*

### Objectives:
1. Ingest raw Online Retail II transactions across two sheets (2009-2010 and 2010-2011).
2. Audit missing values, anomalies, and cancellations.
3. Filter non-positive quantities and prices.
4. Clean and impute Customer IDs, remove duplicate entries.
5. Compute `TotalAmount` and date components, winsorize extreme outliers.
6. Export validated dataset as CSV and Parquet.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Imports ready.')

Imports ready.


In [2]:
raw_path = os.path.join('..', '..', 'data', 'raw', 'online_retail_II.xlsx')
print(f'Reading {raw_path}...')

df1 = pd.read_excel(raw_path, sheet_name='Year 2009-2010')
df2 = pd.read_excel(raw_path, sheet_name='Year 2010-2011')

col_map = {'Customer ID': 'CustomerID'}
df1 = df1.rename(columns=col_map)
df2 = df2.rename(columns=col_map)

df = pd.concat([df1, df2], ignore_index=True)
print(f'Total combined raw records: {len(df):,}')
df.head()

Reading ..\..\data\raw\online_retail_II.xlsx...


Total combined raw records: 1,067,371


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Data Cleaning Pipeline
print('Initial Null Values:\n', df.isnull().sum())

# 1. Filter out cancellations (Invoices starting with 'C')
is_cancelled = df['Invoice'].astype(str).str.startswith('C')
df_clean = df[~is_cancelled].copy()
print(f'Cancelled records removed: {is_cancelled.sum():,}')

# 2. Filter positive quantity & price
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# 3. Handle missing customer IDs
df_clean = df_clean[df_clean['CustomerID'].notnull()].copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 4. Remove duplicate transactions
df_clean = df_clean.drop_duplicates()

# 5. Computed Fields & Date Parsing
df_clean['TotalAmount'] = (df_clean['Quantity'] * df_clean['Price']).round(2)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Date'] = df_clean['InvoiceDate'].dt.date
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

print(f'\nFinal Cleaned Dataset: {len(df_clean):,} transactions across {df_clean["CustomerID"].nunique():,} customers.')

Initial Null Values:
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
CustomerID     243007
Country             0
dtype: int64


Cancelled records removed: 19,494



Final Cleaned Dataset: 779,425 transactions across 5,878 customers.


In [4]:
# Summary Statistics of Cleaned Features
display_cols = ['Quantity', 'Price', 'TotalAmount']
print(df_clean[display_cols].describe().round(2))

# Export Artifacts
out_parquet = os.path.join('..', '..', 'data', 'processed', 'cleaned_transactions.parquet')
out_csv = os.path.join('..', '..', 'data', 'processed', 'cleaned_transactions.csv')

df_clean['Invoice'] = df_clean['Invoice'].astype(str)
df_clean['StockCode'] = df_clean['StockCode'].astype(str)
df_clean['Description'] = df_clean['Description'].fillna('').astype(str)
df_clean['Country'] = df_clean['Country'].fillna('Unknown').astype(str)
df_clean['Date'] = df_clean['Date'].astype(str)

df_clean.to_parquet(out_parquet, index=False)
print(f'Saved cleaned data to {out_parquet}')

        Quantity      Price  TotalAmount
count  779425.00  779425.00    779425.00
mean       13.49       3.22        22.29
std       145.86      29.68       227.43
min         1.00       0.00         0.00
25%         2.00       1.25         4.95
50%         6.00       1.95        12.48
75%        12.00       3.75        19.80
max     80995.00   10953.50    168469.60


Saved cleaned data to ..\..\data\processed\cleaned_transactions.parquet
